In [1]:
import os
import yaml
import datetime
import numpy as np
import matplotlib.pyplot as plt

from AgeNet import environments, Agent, AgentsSimulator
from AgeNet.environments import physics
from AgeNet import analyses, experiments
from AgeNet.learning import RLBrain


In [7]:
# 1. Environment ------------------------------------------------------------------------
# ---------------------------------------------------------------------------------------
L = 45                                  # The length of the simulation box
buildings_type = "random"               # Type of placement of buildings (random or regular)
num_buildings  = 00                     # Number of buildings in environment
num_streets    = 0                      # Number of streets in environment

sumo_cfg    = "./Sumo/SumoScenario/simulation.sumocfg"
sumo_binary = "/nix/store/gp4hbfbrxg4zzjs7g3nl7vy0rw4jmm2y-sumo-1.22.0/bin/sumo"
step_length = 1.0                       # 
use_gui     = False                     # 
sumo_seed   = None                      # 

environment_parameters = [L, buildings_type, num_buildings, num_streets]            # manual env
# environment_parameters = [sumo_cfg, sumo_binary, step_length, use_gui, sumo_seed]   # sumo env


# 2. Agent Learning ---------------------------------------------------------------------
# ---------------------------------------------------------------------------------------
N = 100                                 # Number of agents
Alphas = [-0.5, 0.3, 1.0, -1000]       # Hamiltonian constant coefficients

state_dim     = 3                       # Dimensions of the agent state for neural network input
action_dim    = 2                       # Neural network output dimensions (Q values)
learning_rate = 1e-5                    # Learning rate in model training
discount_rate = 0.98                    # The effect of future decisions on current decisions
batch_size    = 20                      # Selected package of replay_memory to perform training
steps_per_train = 10                    # The training function is called once every ... steps
Initialize_model_path = f'./Trained Models/20260213-161716/model_minHamiltonian.keras'
brain_parameters = [state_dim, action_dim, learning_rate, discount_rate, batch_size, Initialize_model_path]


# 3. Communication ----------------------------------------------------------------------
# ---------------------------------------------------------------------------------------
absorption  = 0.5
base_delay  = 0.01
max_delay   = 1.0
communication_parameters = [absorption, base_delay, max_delay]


# 4. Action Policy ----------------------------------------------------------------------
# ---------------------------------------------------------------------------------------
action_policy_type = 'kNN'               # FR or kNN or MD or RL
action_k_min = 5
policy_parameters = [action_policy_type, action_k_min]


# 5. Functions --------------------------------------------------------------------------
# ---------------------------------------------------------------------------------------
moving     = False                       # Enable the move function to agents
requesting = False                       # Enable the sending request function
training   = False                       # Enable each agent model training function
Functions  = [moving, requesting, training]


# 6. Directory and Seed -----------------------------------------------------------------
# ---------------------------------------------------------------------------------------
log_dir = f"Results/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
seed = 30
np.random.seed(seed)



# Configs -------------------------------------------------------------------------------
# ---------------------------------------------------------------------------------------
config = {
    "environment": environment_parameters,

    'number of agent': N,
    "brain": brain_parameters,
    'alpha': Alphas,
    'communication': communication_parameters,
    'action policy': policy_parameters,
    'functions': Functions,
    'seed': seed
}



### main runs

In [9]:
def main(config, log_dir):
    # Parameters ------------------------------------------------------------------------------------
    N                        = config['number of agent']
    environment_parameters   = config['environment']
    brain_parameters         = config['brain']
    Alphas                   = config['alpha']
    communication_parameters = config['communication']
    policy_parameters        = config['action policy']
    Functions                = config['functions']
    seed                     = config['seed']

    # Init ------------------------------------------------------------------------------------------
    np.random.seed(seed)
    environment = environments.SimpleEnvironment(*environment_parameters)

    Agents = []
    for i in range(N): 
        id = f'{i}'

        movement = physics.NonMarkovian(environment)
        environment.add_entity(id, movement)
        entity = environment.get_entity(id)
        agent = Agent(id, entity, Alphas, brain_parameters, communication_parameters, policy_parameters)
        Agents.append(agent)

    simulator  = AgentsSimulator(environment, Agents, Functions)
    visualizer = analyses.NetworkVisualizer(environment)
    result_exporter = experiments.ResultExporter(environment)
    result_ploter   = experiments.ResultPloter(log_dir, environment)


    # Run -------------------------------------------------------------------------------------------
    for step in range(1001):
        simulator.run(Agents, steps_per_train=10)

        # -------------------------------------------------------------------------------------------
        result_exporter.collect(step, Agents)
        if step%5 == 0: visualizer.draw(Agents, step)
        print("\rStep: {}".format(step), end="")

    if not os.path.exists(log_dir): os.makedirs(log_dir)

    result_exporter.export_excel(f'./{log_dir}', f'{seed}, Policy Type = {policy_parameters[0]}, Requesting = {Functions[1]}')
    result_ploter.average_result(result_exporter.averaged_results)
    anim = visualizer.camera.animate(interval= 120, repeat=True, repeat_delay= 500, blit=False)
    anim.save(f'./{log_dir}/Animation.gif')
    with open(f"{log_dir}/config.yaml", "w") as f: yaml.dump(config, f, sort_keys=False)
    

In [ ]:
Functions_ = [ [True, False, False], [True, False, True], [True, True, True] ]  # [moving, requesting, training]
Policy_Parameters_ = [ ['MD', 5], ]                        # [action_policy_type, action_k_min]
# log_dir = './kNN/Static'

for seed in range(30, 31):
    for init in range(1):
        Functions = Functions_[init]
        policy_parameters = Policy_Parameters_[init]

        config = config = {
            "environment": environment_parameters,

            'number of agent': N,
            "brain": brain_parameters,
            'alpha': Alphas,
            'communication': communication_parameters,
            'action policy': policy_parameters,
            
            'functions': Functions,
            'seed': seed
        }

        main(config, log_dir)
        print(f'\t seed = {seed}, Policy Type = {policy_parameters[0]}, Requesting = {Functions[1]}')
    

### learning

In [ ]:
brain_parameters = [state_dim, action_dim, learning_rate, discount_rate, batch_size]
brain = RLBrain(*brain_parameters)

environment = environments.SimpleEnvironment(*environment_parameters)

Agents = []
for i in range(N): 
    id = f'{i}'

    movement = physics.NonMarkovian(environment)
    environment.add_entity(id, movement)
    entity = environment.get_entity(id)
    agent = Agent(id, entity, Alphas, brain_parameters, communication_parameters, policy_parameters)
    agent.brain = brain
    Agents.append(agent)

Functions  = [False, False, training]
simulator  = AgentsSimulator(environment, Agents, Functions)

visualizer = analyses.NetworkVisualizer(environment)
result_exporter = experiments.ResultExporter(environment)
result_ploter   = experiments.ResultPloter(log_dir, environment)


In [ ]:
best_H, best_G, best_H2G2 = 0, 0, 0
for step in range(1001):
    simulator.run(Agents, steps_per_train=1)

    # -------------------------------------------------------------------------------------------
    result_exporter.collect(step, Agents)
    if step%5 == 0: visualizer.draw(Agents, step)
    print("\rStep: {}".format(step), end="")

    # Find Best Weights ------------------------------------------------------------------------- 
    step, hamilton, giant, _, _, _, _ = result_exporter.averaged_results[-1]    
    if step > 500:
        if hamilton <= best_H:                                  # find the minimum of Hamiltonian
            H_best_weight    = Agents[0].model.get_weights()    # saving model weights for the best Hamiltonian
            best_H = hamilton

        if giant >= best_G:
            G_best_weight    = Agents[0].model.get_weights()    # saving model weights for the best Giant Component
            best_G = giant

        if np.sqrt(hamilton**2 + (giant/1e3)**2) >= best_H2G2:
            H2G2_best_weight = Agents[0].model.get_weights()
            best_H2G2 = np.sqrt(hamilton**2 + (giant/1e3)**2)


if not os.path.exists(log_dir): os.makedirs(log_dir)

anim = visualizer.camera.animate(interval= 120, repeat=True, repeat_delay= 500, blit=False)
result_exporter.export_excel(f'./{log_dir}')
result_ploter.average_result(result_exporter.averaged_results)
anim.save(f'./{log_dir}/Animation - Training.gif')
with open(f"{log_dir}/config.yaml", "w") as f: yaml.dump(config, f, sort_keys=False)


last_weight = Agents[0].model.get_weights()

best_models = [H_best_weight, G_best_weight, H2G2_best_weight, last_weight]
best_models_name = ['minHamiltonian', 'maxConnectivity', 'balancedHG', 'FinalStep']
for i, best_model in enumerate(best_models):
    model_ = Agents[0].model
    model_.set_weights(best_model)
    model_.save(f'{log_dir}/model_{best_models_name[i]}.keras')
    

### Simple Environment

In [ ]:
environment = environments.SimpleEnvironment(*environment_parameters)

Agents = []
for i in range(N): 
    id = f'{i}'

    movement = physics.NonMarkovian(environment)
    environment.add_entity(id, movement)
    entity = environment.get_entity(id)
    agent = Agent(id, entity, Alphas, brain_parameters, communication_parameters, policy_parameters)
    Agents.append(agent)

simulator  = AgentsSimulator(environment, Agents, Functions)

visualizer = analyses.NetworkVisualizer(environment)
result_exporter = experiments.ResultExporter(environment)
result_ploter   = experiments.ResultPloter(log_dir, environment)


In [ ]:
for step in range(1001):
    simulator.run(Agents, steps_per_train=10)

    # -------------------------------------------------------------------------------------------
    result_exporter.collect(step, Agents)
    if step%5 == 0: visualizer.draw(Agents, step)
    print("\rStep: {}".format(step), end="")

if not os.path.exists(log_dir): os.makedirs(log_dir)

anim = visualizer.camera.animate(interval= 120, repeat=True, repeat_delay= 500, blit=False)
result_exporter.export_excel(f'./{log_dir}')
result_ploter.average_result(result_exporter.averaged_results)
anim.save(f'./{log_dir}/Animation.gif')


### Sumo Environment

In [ ]:
environment = environments.SumoEnvironment(*environment_parameters)
environment.start()

Agents = []

simulator  = AgentsSimulator(environment, Agents, Functions)
visualizer = analyses.NetworkVisualizer(environment)
result_exporter = experiments.ResultExporter(environment)
result_ploter   = experiments.ResultPloter(log_dir, environment)


In [ ]:
ids, ids_prev = [], []

for step in range(1001):
    environment.step()
    ids      = environment.get_entity_ids()
    new_ids  = list(set(ids) - set(ids_prev))
    del_ids  = list(set(ids_prev) - set(ids))
    ids_prev = ids

    for id in new_ids:
        entity = environment.get_entity(id)
        agent = Agent(id, entity, Alphas, brain_parameters, communication_parameters)
        Agents.append(agent)
        simulator.add_agent(agent, Functions)

    for id in del_ids:
        removed_agent = next((agent for agent in Agents if agent.id == id), None)
        Agents.remove(removed_agent)
        simulator.remove_agent(removed_agent)
    

    simulator.run(Agents, steps_per_train=10)
    
    # -------------------------------------------------------------------------------------------
    if len(ids) > 1:
        result_exporter.collect(step, Agents)
        if step%5 == 0: visualizer.draw(Agents, step)
    print("\rStep: {}".format(step), end="")

environment.close()
if not os.path.exists(log_dir): os.makedirs(log_dir)

anim = visualizer.camera.animate(interval= 120, repeat=True, repeat_delay= 500, blit=False)
result_exporter.export_excel(f'./{log_dir}')
result_ploter.average_result(result_exporter.averaged_results)
anim.save(f'./{log_dir}/test.gif')

d:\Applications\Anaconda3\Lib\site-packages\keras\src\models\functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: input. Received: the structure of inputs={'input': '*'}
  warnings.warn(


Step: 1000

### Tests